# 発展課題：計算・長さ・埋め込み
共通課題で学習したモデルを保存し、自分の問いを立てて取り組む。

## 準備
先に `environment-setup.md` に従って、教員が指定したGitHubリポジトリから教材を取得し、Google Driveに置く。
ColabではGPUランタイムを選び、以下を上から実行する。

PythonファイルはローカルのCursor / VS Codeで編集・保存し、Driveの同期完了を待ってからColabで実行する。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/project-research

In [ ]:
%%capture
%pip install -r requirements.txt

`configs/addition/train.yaml`の`train.wandb`にある`project`・`group`・`name`を設定する．
APIキーはファイルに書かず，次のセルでログインする．

In [ ]:
import wandb
wandb.login()

## 1. 自分で計算課題を設定する（個人課題）

[src/addition.py](../src/addition.py)の`sum_problem`で問題と正解を定義し，`format_problem`で入出力の表記を定義する．
入出力例を3つ，採用する表記とその理由，成功の判断基準を書く．

必要に応じて[lexer.yaml](../configs/addition/lexer.yaml)の語彙と，[src/train.py](../src/train.py)の評価時の生成長を変更する．
比較する実験ごとに[train.yaml](../configs/addition/train.yaml)の保存先とW&Bの実行名を変え，使用したコードと設定も保存しておく．

In [ ]:
from src.addition import sum_problem, format_problem

problem, answer = sum_problem(0)
print(problem, answer)
print(format_problem(problem, answer))

In [ ]:
!bash scripts/addition.sh

## 2. 未知の桁数でも計算できるか？

共通課題で学習したモデルを，再学習せずに評価する．
評価の使い方と図の表示は[評価Notebook](evaluation.ipynb)で確認できる．
[configs/addition/evaluate.yaml](../configs/addition/evaluate.yaml)で，モデルの保存先・表記関数・lexer・評価する桁数を指定する．

- `model_path`：学習済みのモデルとtokenizerを保存したフォルダ．
- `format_function`：学習時の表記関数を`モジュール名:関数名`で指定する．例えば`src.addition:format_problem`．
- `answer_function`：正解を計算する関数．加算は`operator:add`，乗算は`operator:mul`．
- `lexer_config`：学習時に使ったlexerの設定ファイル．
- `digits`：評価する桁数．各桁数の全組合せで評価する．
- `samples_per_cell`：桁数の組ごとの問題数．
- `max_length`：答えを生成するtoken数の上限．終了tokenの分も含める．
- `output_dir`：結果の保存先．

学習後に`format_problem`や埋め込みの実装を変更した場合，評価には学習時の実装を使う．表記関数を別のPythonファイルに保存した場合は，そのモジュール名を指定する．lexerも学習時の設定を残しておく．

評価・描画は[src/evaluate.py](../src/evaluate.py)から実行する．桁数は0埋めや逆順にする前の整数の桁数で集計する．終了tokenまで含めて，生成した答えと変換後の正解が一致した割合を求める．

In [ ]:
!bash scripts/evaluate.sh

評価結果は`predictions.csv`，桁数ごとの集計は`accuracy_by_digits.csv`に保存される．
ヒートマップは行が第1オペランドの桁数，列が第2オペランドの桁数．折れ線は両方が同じ桁数の問題の完全一致率を表す．

> 演習

1. 以下のセルで2つの図を表示せよ．
2. 桁数ごとの結果と，具体的な誤答を記録せよ．

In [ ]:
from IPython.display import Image, display
from omegaconf import OmegaConf

cfg = OmegaConf.load('configs/addition/evaluate.yaml')
display(Image(filename=f'{cfg.output_dir}/length_heatmap.png'))
display(Image(filename=f'{cfg.output_dir}/length_curve.png'))

図だけを描き直す場合は，保存済みのCSVを使う．

In [ ]:
from support.length_evaluation import plot_results

plot_results(f'{cfg.output_dir}/predictions.csv')

## 3. 数字の埋め込みを工夫する
編集：[embeddings.py](../src/embeddings.py)の`NumberEmbedding`（D1）。
入力はtoken IDの行列`(batch, length)`、出力はベクトル`(batch, length, d_model)`。

通常の埋め込みは数字を別々の記号として扱う。数字の値に比例した成分を加えるとどうなるだろう？
`values`は数字tokenを0〜1の値へ写し、演算記号などを0にする。
数字の実際のtoken IDはtokenizerから渡される。

まずコードを読み、0・1・9・演算記号で出力のどの部分が変わるか予想する。
この例で数字間の距離が値の差に必ず比例するわけではないことにも注意する。

In [ ]:
import torch
from src.embeddings import NumberEmbedding

embedding = NumberEmbedding(vocab_size=10, d_model=16, digit_ids=list(range(10)))
input_ids = torch.tensor([[0, 1, 9]])
embedding(input_ids)

[src/embeddings.py](../src/embeddings.py)の登録名を`train.yaml`の`model.input_embedding_type`に指定する．
CALT 1.5.0のdecoder-onlyでは，[support/model.py](../support/model.py)の`connect_embeddings`で入力埋め込みを接続する．
`src/train.py`でモデルを作った直後に，以下を追加する．

```python
from support.model import connect_embeddings
connect_embeddings(model, cfg.model, io['tokenizer'])
```

同じデータ・表記・学習回数で通常の埋め込みと比較する．
加算で比較する場合は，`sum_problem`と`format_problem`を加算の定義に戻す．

In [ ]:
!bash scripts/addition.sh

## 4. 桁位置を表す：Abacus
編集：[embeddings.py](../src/embeddings.py)の`AbacusEmbedding`（D2・D3）。
数字列ごとに1, 2, 3…と位置を数え、演算記号でリセットする。
学習時は数字位置にランダムなoffsetを足し、評価時は固定する。

入力・答えとも逆順にすると、同じ位に同じ位置番号が付く。
次の例で、演算記号や数字列の境界の位置番号を確かめる。
参考：[McLeish et al.](https://arxiv.org/abs/2405.17399)。

In [ ]:
from src.embeddings import AbacusEmbedding

embedding = AbacusEmbedding(vocab_size=11, d_model=16, digit_ids=list(range(10)))
embedding.eval()
input_ids = torch.tensor([[2, 1, 10, 8]])  # 10は演算記号のtoken ID
embedding.position_ids(input_ids)

`train.yaml`の`model.input_embedding_type`を`abacus`，`model.use_positional_embedding`を`none`に設定する．
前節と同じ`connect_embeddings`でモデルに接続する．
`format_problem`で表記を設定し，通常の埋め込みと同じデータ・表記・学習回数で比較する．

In [ ]:
!bash scripts/addition.sh

保存した各モデルを，課題2と同じ評価コードで比較する．モデルごとに設定ファイルの`model_path`・表記関数・結果の保存先を変更する．

## 報告
- 問いと予想
- 編集した関数と実験条件
- ID/OODの結果、W&Bリンク、具体的な誤答
- 支持された説明と、まだわからないこと

まずIDで十分に学習できたか確認し、学習失敗と外挿失敗を区別する。
必要ならseedを変え、良いseedだけを選ばない。
OODの結果を見ながら調整した評価は開発用とし、最後の確認用の問題と分ける。
結果はGoogle Driveの教材フォルダ内の `runs/` に保存される。